<a href="https://colab.research.google.com/github/melanieyes/nlp-journey/blob/master/word2vec_rnn_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, I explore how different neural architectures understand language, starting from simple distributional models to more advanced sequence models.

The central idea behind all of these methods is the **distributional hypothesis**:  
*“You shall know a word by the company it keeps.”*

I begin with **Word2Vec**, which learns word representations through local context:

- **CBOW (Continuous Bag of Words)** predicts a word from its surrounding context.  
- **Skip-gram** does the reverse: it predicts surrounding words from a center word.  

These models treat language as a collection of local co-occurrences, without modeling sequence dynamics explicitly.

Then move to **Recurrent Neural Networks (RNNs)**, which introduce the notion of **temporal dependency**.  
Unlike Word2Vec, RNNs process text **sequentially**, maintaining a hidden state that evolves over time. This allows the model to capture how meaning accumulates across a sentence.

Finally, I explore **Transformers**, which replace recurrence with **self-attention**.  
Instead of processing tokens one by one, Transformers allow each word to directly attend to all other words in the sequence, enabling richer and more flexible representations of context.

Rather than building highly optimized models, the goal here is to:

- Understand **how each architecture works internally**  
- Compare how they represent and process the same sentence  
- Build intuition for:
  - context vs sequence  
  - local vs global information  
  - sequential vs parallel computation  

I will use a simple example sentence:

> "a cute teddy bear is reading"

to illustrate how each model:
- encodes words  
- propagates information  
- and makes predictions  

As you go through the notebook, pay attention to:

- How **CBOW and Skip-gram** rely purely on local context  
- How **RNN hidden states evolve over time**  
- How **Transformer attention distributes focus across words**  

Each section includes minimal implementations designed to expose the **core mechanics**, rather than hide them behind abstraction.



In [1]:
sentence = "A cute teddy bear is reading".lower().split()
print(sentence)


['a', 'cute', 'teddy', 'bear', 'is', 'reading']


In [2]:
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

print(word_to_idx)

{'a': 0, 'bear': 1, 'cute': 2, 'is': 3, 'reading': 4, 'teddy': 5}


In [3]:
import numpy as np

sentence = "A cute teddy bear is reading".lower().split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
embed_dim = 4

np.random.seed(42)

# input embedding matrix
W_in = np.random.randn(vocab_size, embed_dim) * 0.1

# output embedding matrix
W_out = np.random.randn(embed_dim, vocab_size) * 0.1

# Example: predict "teddy" from ["a", "cute", "bear", "is"]
context_words = ["a", "cute", "bear", "is"]
target_word = "teddy"

context_ids = [word_to_idx[w] for w in context_words]
target_id = word_to_idx[target_word]

# 1. get embeddings of context words
context_embeds = W_in[context_ids]                  # shape: (4, embed_dim)

# 2. average them
h = np.mean(context_embeds, axis=0, keepdims=True)  # shape: (1, embed_dim)

# 3. project to vocabulary scores
logits = h @ W_out                                  # shape: (1, vocab_size)

# 4. softmax
exp_logits = np.exp(logits - np.max(logits))
probs = exp_logits / np.sum(exp_logits)

pred_id = np.argmax(probs)
pred_word = idx_to_word[pred_id]

print("CBOW context:", context_words)
print("CBOW target :", target_word)
print("CBOW predicted word:", pred_word)
print("Probabilities:")
for i, p in enumerate(probs[0]):
    print(idx_to_word[i], "->", round(float(p), 4))

CBOW context: ['a', 'cute', 'bear', 'is']
CBOW target : teddy
CBOW predicted word: teddy
Probabilities:
a -> 0.1672
bear -> 0.1653
cute -> 0.166
is -> 0.1672
reading -> 0.166
teddy -> 0.1683


Skip-gram


In [4]:
import numpy as np

sentence = "A cute teddy bear is reading".lower().split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)
embed_dim = 4

np.random.seed(42)

W_in = np.random.randn(vocab_size, embed_dim) * 0.1
W_out = np.random.randn(embed_dim, vocab_size) * 0.1

center_word = "teddy"
context_words = ["a", "cute", "bear", "is"]

center_id = word_to_idx[center_word]

# 1. embedding of center word
h = W_in[center_id:center_id+1]   # shape: (1, embed_dim)

# 2. scores for all words
logits = h @ W_out                # shape: (1, vocab_size)

# 3. softmax
exp_logits = np.exp(logits - np.max(logits))
probs = exp_logits / np.sum(exp_logits)

print("Skip-gram center word:", center_word)
print("True context words:", context_words)
print("Predicted probability for each vocab word:")
for i, p in enumerate(probs[0]):
    print(idx_to_word[i], "->", round(float(p), 4))

Skip-gram center word: teddy
True context words: ['a', 'cute', 'bear', 'is']
Predicted probability for each vocab word:
a -> 0.1659
bear -> 0.1668
cute -> 0.1673
is -> 0.1698
reading -> 0.1661
teddy -> 0.164


RNN

In [5]:
import numpy as np

sentence = "A cute teddy bear is reading".lower().split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

embed_dim = 4
hidden_dim = 5

np.random.seed(42)

# embeddings
E = np.random.randn(vocab_size, embed_dim) * 0.1

# RNN parameters
Wx = np.random.randn(hidden_dim, embed_dim) * 0.1
Wh = np.random.randn(hidden_dim, hidden_dim) * 0.1
b = np.zeros((hidden_dim, 1))

h = np.zeros((hidden_dim, 1))

print("RNN step-by-step hidden states:\n")

for word in sentence:
    x = E[word_to_idx[word]].reshape(-1, 1)   # embedding column vector
    h = np.tanh(Wx @ x + Wh @ h + b)
    print(f"word = {word:8s} | hidden = {np.round(h.T, 3)}")

RNN step-by-step hidden states:

word = a        | hidden = [[-0.005  0.022 -0.012 -0.002 -0.002]]
word = cute     | hidden = [[ 0.006 -0.003 -0.001 -0.007  0.002]]
word = teddy    | hidden = [[-0.016 -0.035  0.021  0.005  0.014]]
word = bear     | hidden = [[-0.009  0.008 -0.    -0.018 -0.011]]
word = is       | hidden = [[0.013 0.007 0.013 0.062 0.   ]]
word = reading  | hidden = [[ 0.015 -0.022  0.006 -0.005  0.006]]


Transformer

In [6]:
import numpy as np

sentence = "A cute teddy bear is reading".lower().split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

embed_dim = 4
d_k = 4

np.random.seed(42)

# token embeddings
E = np.random.randn(vocab_size, embed_dim) * 0.1
X = np.array([E[word_to_idx[w]] for w in sentence])   # shape: (seq_len, embed_dim)

# attention weight matrices
WQ = np.random.randn(embed_dim, d_k) * 0.1
WK = np.random.randn(embed_dim, d_k) * 0.1
WV = np.random.randn(embed_dim, d_k) * 0.1

# compute Q, K, V
Q = X @ WQ
K = X @ WK
V = X @ WV

# attention scores
scores = Q @ K.T / np.sqrt(d_k)

# softmax row-wise
exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attn_weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# output
output = attn_weights @ V

print("Words:", sentence)
print("\nAttention matrix:")
print(np.round(attn_weights, 3))

print("\nTransformer output vectors:")
print(np.round(output, 3))

Words: ['a', 'cute', 'teddy', 'bear', 'is', 'reading']

Attention matrix:
[[0.167 0.167 0.167 0.167 0.167 0.167]
 [0.167 0.167 0.167 0.167 0.167 0.167]
 [0.167 0.167 0.167 0.167 0.167 0.167]
 [0.167 0.167 0.167 0.167 0.167 0.167]
 [0.167 0.167 0.167 0.167 0.167 0.167]
 [0.167 0.167 0.167 0.167 0.167 0.167]]

Transformer output vectors:
[[-0.001  0.     0.002 -0.001]
 [-0.001  0.     0.002 -0.001]
 [-0.001  0.     0.002 -0.001]
 [-0.001  0.     0.002 -0.001]
 [-0.001  0.     0.002 -0.001]
 [-0.001  0.     0.002 -0.001]]


In [7]:
sentence = "a cute teddy bear is reading".split()

# CBOW example
context = ["a", "cute", "is", "reading"]
target = "teddy"
print("CBOW:")
print("input context ->", context)
print("predict center ->", target)

# Skip-gram example
center = "teddy"
contexts = ["a", "cute", "bear", "is"]
print("\nSkip-gram:")
print("input center ->", center)
print("predict context ->", contexts)

# RNN example
print("\nRNN:")
hidden = "start"
for w in sentence:
    print(f"read '{w}' -> update hidden state")

# Transformer example
print("\nTransformer:")
for w in sentence:
    print(f"word '{w}' attends to all words in the sentence")

CBOW:
input context -> ['a', 'cute', 'is', 'reading']
predict center -> teddy

Skip-gram:
input center -> teddy
predict context -> ['a', 'cute', 'bear', 'is']

RNN:
read 'a' -> update hidden state
read 'cute' -> update hidden state
read 'teddy' -> update hidden state
read 'bear' -> update hidden state
read 'is' -> update hidden state
read 'reading' -> update hidden state

Transformer:
word 'a' attends to all words in the sentence
word 'cute' attends to all words in the sentence
word 'teddy' attends to all words in the sentence
word 'bear' attends to all words in the sentence
word 'is' attends to all words in the sentence
word 'reading' attends to all words in the sentence


With Pytorch

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

sentence = "a cute teddy bear is reading".split()

vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

print("Sentence:", sentence)
print("Vocab:", word_to_idx)

Sentence: ['a', 'cute', 'teddy', 'bear', 'is', 'reading']
Vocab: {'a': 0, 'bear': 1, 'cute': 2, 'is': 3, 'reading': 4, 'teddy': 5}


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

sentence = "a cute teddy bear is reading".split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, context_ids):
        embeds = self.embedding(context_ids)      # [C, D]
        hidden = embeds.mean(dim=0)               # [D]
        logits = self.linear(hidden)              # [V]
        return logits

model = CBOW(len(vocab), embed_dim=16)
optimizer = optim.Adam(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

context_words = ["a", "cute", "is", "reading"]
target_word = "teddy"

context_ids = torch.tensor([word_to_idx[w] for w in context_words], dtype=torch.long)
target_id = torch.tensor([word_to_idx[target_word]], dtype=torch.long)

for epoch in range(100):
    optimizer.zero_grad()
    logits = model(context_ids).unsqueeze(0)   # [1, V]
    loss = criterion(logits, target_id)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        pred = torch.argmax(logits, dim=1).item()
        print(f"Epoch {epoch:3d} | Loss={loss.item():.4f} | Pred={idx_to_word[pred]}")

final_logits = model(context_ids)
final_pred = torch.argmax(final_logits).item()

print("\nFinal CBOW prediction:", idx_to_word[final_pred])

Epoch   0 | Loss=1.5806 | Pred=teddy
Epoch  20 | Loss=0.0000 | Pred=teddy
Epoch  40 | Loss=0.0000 | Pred=teddy
Epoch  60 | Loss=0.0000 | Pred=teddy
Epoch  80 | Loss=0.0000 | Pred=teddy

Final CBOW prediction: teddy


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

sentence = "a cute teddy bear is reading".split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, center_id):
        # center_id: [1]
        embed = self.embedding(center_id)             # [1, D]
        logits = self.linear(embed).squeeze(0)        # [V]
        return logits

model = SkipGram(vocab_size=len(vocab), embed_dim=8)

center_word = "teddy"
true_context_words = ["a", "cute", "bear", "is"]

center_id = torch.tensor([word_to_idx[center_word]], dtype=torch.long)
logits = model(center_id)
probs = F.softmax(logits, dim=0)

print("Skip-gram")
print("Center word:", center_word)
print("True context words:", true_context_words)
print("Probabilities:", {idx_to_word[i]: round(probs[i].item(), 4) for i in range(len(vocab))})

Skip-gram
Center word: teddy
True context words: ['a', 'cute', 'bear', 'is']
Probabilities: {'a': 0.2922, 'bear': 0.1215, 'cute': 0.1237, 'is': 0.257, 'reading': 0.0947, 'teddy': 0.1109}


In [11]:
import torch
import torch.nn as nn

sentence = "a cute teddy bear is reading".split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

    def forward(self, x):
        # x: [batch, seq_len]
        emb = self.embedding(x)       # [B, T, D]
        output, hidden = self.rnn(emb)
        return output, hidden

model = SimpleRNN(vocab_size=len(vocab), embed_dim=8, hidden_dim=10)

x = torch.tensor([[word_to_idx[w] for w in sentence]], dtype=torch.long)  # [1, seq_len]

output, hidden = model(x)

print("RNN")
print("Input ids shape:", x.shape)
print("Output shape:", output.shape)   # [1, seq_len, hidden_dim]
print("Final hidden shape:", hidden.shape)  # [1, 1, hidden_dim]

for i, word in enumerate(sentence):
    print(f"Step {i}: word='{word}', hidden state={output[0, i].detach().numpy().round(3)}")

RNN
Input ids shape: torch.Size([1, 6])
Output shape: torch.Size([1, 6, 10])
Final hidden shape: torch.Size([1, 1, 10])
Step 0: word='a', hidden state=[ 0.052  0.141 -0.624 -0.132  0.254 -0.336 -0.629  0.411 -0.572  0.393]
Step 1: word='cute', hidden state=[ 0.602  0.703  0.033 -0.405  0.675 -0.666 -0.462  0.369  0.143 -0.444]
Step 2: word='teddy', hidden state=[ 0.003 -0.085 -0.043  0.057 -0.014  0.032  0.042 -0.495  0.477  0.466]
Step 3: word='bear', hidden state=[ 0.087  0.429  0.73   0.478 -0.139 -0.323  0.152  0.833 -0.147 -0.452]
Step 4: word='is', hidden state=[ 0.596 -0.325  0.057  0.181  0.701 -0.887 -0.421 -0.14  -0.959  0.811]
Step 5: word='reading', hidden state=[-0.113  0.625  0.871  0.843 -0.411  0.657  0.741  0.833 -0.447  0.882]


In [12]:
import torch
import torch.nn as nn

sentence = "a cute teddy bear is reading".split()
vocab = sorted(set(sentence))
word_to_idx = {w: i for i, w in enumerate(vocab)}

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, x):
        # x: [batch, seq_len]
        emb = self.embedding(x)                    # [B, T, D]
        attn_output, attn_weights = self.attn(emb, emb, emb)
        return attn_output, attn_weights

model = TinyTransformer(vocab_size=len(vocab), embed_dim=8, num_heads=2)

x = torch.tensor([[word_to_idx[w] for w in sentence]], dtype=torch.long)

attn_output, attn_weights = model(x)

print("Transformer")
print("Attention output shape:", attn_output.shape)   # [1, seq_len, embed_dim]
print("Attention weights shape:", attn_weights.shape) # [1, seq_len, seq_len]

print("\nAttention matrix:")
print(attn_weights[0].detach().numpy().round(3))

Transformer
Attention output shape: torch.Size([1, 6, 8])
Attention weights shape: torch.Size([1, 6, 6])

Attention matrix:
[[0.073 0.133 0.198 0.267 0.152 0.177]
 [0.072 0.148 0.265 0.251 0.087 0.178]
 [0.289 0.196 0.146 0.155 0.079 0.135]
 [0.206 0.205 0.151 0.14  0.146 0.153]
 [0.234 0.207 0.104 0.118 0.193 0.145]
 [0.191 0.244 0.159 0.139 0.092 0.175]]
